<div style="border-left:4px solid #f472b6;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#f472b6;">Optimization</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Five variants of the hybrid design, benchmarked and ranked.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">Five ways of asking the same pipeline for SQL, over ten questions chosen to span the difficulty range. Every question carries a hand-written reference query, so a variant is judged on the answer it returns and not on agreeing with the other four.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so a fresh clone needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
#
# A version created through the API cannot read that secret - Kaggle answers 400
# no matter how the box is ticked in the editor, and the attachment cannot be
# declared in kernel-metadata.json either (Kaggle/kaggle-cli#582). Notebook 1's
# output carries the whole repository, so fall back to that copy. Only notebook 1
# has no input to fall back to, and only notebook 1 has to be saved from the
# browser rather than pushed.
import os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")


def clone_from_github() -> str:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)
    # git writes the clone URL into .git/config, token and all, and Kaggle saves
    # .git with the notebook output. Put the plain address back immediately.
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", "https://github.com/Kirazul/NL2SQL-demo.git"], check=True)
    return "a fresh clone"


def find_in_mounts(relative: str, max_depth: int = 7) -> "list[Path]":
    """Every path under /kaggle/input, so nothing has to guess how deep it is.

    Kaggle has mounted a notebook's output at /kaggle/input/<slug>/ and at
    /kaggle/input/notebooks/<owner>/<slug>/, and the repository sits a further
    level inside that. Each guess at the shape reported a perfectly good output
    as a missing database, so walk for it. Bounded, and never down into the two
    directories that hold the weights and the git objects, because an attached
    output is gigabytes.
    """
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    hits = []
    for dirpath, dirnames, _ in os.walk(root):
        here = Path(dirpath)
        if len(here.relative_to(root).parts) >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if d not in {".git", "models", "wheels"}]
        candidate = here / relative
        if candidate.exists():
            hits.append(candidate)
    return sorted(hits)


def copy_from_setup() -> str:
    # Any pyproject.toml would match, so take the one with the package beside it.
    marker = next((m for m in find_in_mounts("pyproject.toml")
                   if (m.parent / "src" / "nl2sql").is_dir()), None)
    if marker is None:
        return ""
    # Everything except data/ and models/: those are the two gigabytes that get
    # read where they are mounted and are never worth copying.
    shutil.copytree(marker.parent, ROOT,
                    ignore=shutil.ignore_patterns("data", "models", ".git"))
    # The mount is read-only and copytree keeps the modes, but `pip install -e .`
    # writes an egg-info back into the tree.
    subprocess.run(["chmod", "-R", "u+w", str(ROOT)], check=True)
    return f"the copy in {marker.parent}"


if ROOT.exists():
    source = "the working directory"
else:
    try:
        source = clone_from_github()
    except Exception as e:
        source = copy_from_setup()
        if not source:
            # Attached-but-empty and nothing-attached read identically from here,
            # and telling them apart is the whole difficulty, so name what is
            # mounted instead of guessing which one it is.
            mounted = sorted(p.name for p in Path("/kaggle/input").glob("*") if p.is_dir())
            where = ("the attached input(s) " + ", ".join(mounted) + " carry no "
                     "nl2sql/pyproject.toml") if mounted else "no input is attached"
            raise SystemExit(
                f"Nothing to run from: GITHUB_TOKEN could not be read "
                f"({type(e).__name__}: {e}), and {where}. Save this notebook from the "
                "browser, where the secret is readable - or attach a version of NL2SQL 1 "
                "Setup that ran to the end."
            ) from e
        print("GITHUB_TOKEN unreadable, falling back to notebook 1's output:", e)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT, "from", source)

In [ ]:
# pip writes to site-packages, which is not part of notebook 1's saved output, so
# every session installs again. Nearly all of it is already in the Kaggle image.
!pip install -q -e . 2>&1 | tail -2
print("dependencies ready")

In [ ]:
# Notebook 1 built the database, the index and the model weights and saved them
# with its output. Kaggle mounts that output read-only under /kaggle/input, and
# every one of the three is opened read-only here too - so point the settings at
# the mount rather than copying two gigabytes into the working directory.
#
# Where inside the mount they sit is not fixed - Kaggle has used
# /kaggle/input/<slug>/ and /kaggle/input/notebooks/<owner>/<slug>/ - so this
# walks for the database rather than matching a guessed shape. find_in_mounts
# comes from the first cell.
found = find_in_mounts("data/eicu.db")
if not found:
    # Printing the tree beats asserting a cause: the last two guesses at what
    # was wrong here were both wrong, and both would have been settled by this.
    print("nothing matched. What is actually under /kaggle/input:")
    root = Path("/kaggle/input")
    listed = 0
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > 3 or listed > 40:
            dirnames[:] = []
            continue
        print("   " * depth, Path(dirpath).name + "/", " ".join(sorted(filenames)[:6]))
        listed += 1
    raise SystemExit(
        "No data/eicu.db anywhere under /kaggle/input. Notebook 1's saved output is "
        "what carries it: check Input -> Add Input -> Your Work -> NL2SQL 1 Setup, and "
        "that the version pinned there is one that ran to the end."
    )
SETUP = found[0].parents[1]
print("setup output:", SETUP)

# Name a missing piece here rather than several cells later, from inside whichever
# library opens it first.
for path in (SETUP / "data/index.db", SETUP / "models/gliner2-base-v1"):
    if not path.exists():
        print("missing from notebook 1's output:", path)

# Set before nl2sql is imported anywhere: settings() is read once and cached. A
# subprocess started later - the API server in notebook 5 - inherits these too.
os.environ["DB_PATH"] = str(SETUP / "data" / "eicu.db")
os.environ["INDEX_PATH"] = str(SETUP / "data" / "index.db")
os.environ["GLINER_MODEL"] = str(SETUP / "models" / "gliner2-base-v1")
weights = sorted(SETUP.glob("models/*/*.gguf"))
if weights:
    os.environ["LOCAL_GGUF_PATH"] = str(weights[0])

for name in ("DB_PATH", "INDEX_PATH", "GLINER_MODEL", "LOCAL_GGUF_PATH"):
    print(f"  {name:<16} {os.environ.get(name, 'missing - the steps that need it will say so')}")

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
unreadable = []
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        unreadable.append(name)

# All three failing at once is one cause, not three: a version created through the
# API cannot read a notebook secret however the editor shows it, and there is no
# field for the attachment in kernel-metadata.json (Kaggle/kaggle-cli#582). Say so
# once here rather than let every provider step fail separately further down.
if unreadable:
    print("could not read:", ", ".join(unreadable))
    if len(unreadable) == 3:
        print("A version created through the API cannot read notebook secrets. Save this")
        print("notebook from the browser to run the steps that call a provider.")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">1.</span> What is being compared: five methods</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Each moves one variable and shares everything else, so a difference between two rows has exactly one cause.</div></div>

In [ ]:
from nl2sql.optimize.variants import catalogue

print(f"{'method':<11}{'moves':<16}{'starts at':<10}{'strategy':<11}what it does")
print("-" * 112)
for v in catalogue():
    print(f"{v['name']:<11}{v['changes']:<16}{v['model']:<10}{v['strategy']:<11}{v['what']}")

print()
print("strategy is how many calls a method makes:")
print("  single    one call, plus one repair if the query is rejected")
print("  cascade   one call at the chosen rung, climbing only on refusal or low confidence")
print("  consensus three calls at the cheap rung, keeping the answer whose rows agree")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">2.</span> What they run on: the model ladder</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">A rung names where to start, not a single model. When one refuses or errors, the chain falls through to the next.</div></div>

In [ ]:
from nl2sql.config import settings
from nl2sql.llm import cloud
from nl2sql.optimize.benchmark import PRICE_PER_MTOK

cfg = settings()
print(f"{'rung':<9}{'model it starts with':<36}{'$/Mtok in':>10}{'$/Mtok out':>12}")
print("-" * 67)
for rung, name in (("small", cfg.model_small),
                   ("medium", cfg.model_medium),
                   ("large", cfg.model_large)):
    price_in, price_out = PRICE_PER_MTOK.get(name, (0.0, 0.0))
    print(f"{rung:<9}{name:<36}{price_in:>10.2f}{price_out:>12.2f}")

print()
print("the fall-through order from each rung:")
for rung in cloud.LADDER:
    print(f"  {rung:<7} " + " -> ".join(t.name for t in cloud.chain(rung)))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">3.</span> How a question's complexity is scored</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Three counts that stage 1 already produced, weighted. No model is asked, and the score is what cascade routes on.</div></div>

In [ ]:
from nl2sql.optimize.variants import DIFFICULTY_WEIGHTS, HARD_WORDS

print("difficulty = sum over three parts of  weight x min(count / full, 1)")
print()
print(f"{'part':<13}{'weight':>8}{'full at':>9}   counted from")
print("-" * 78)
source = {
    "tables": "tables stage 1 says the question touches",
    "values": "mentions that resolved to a stored value",
    "hard words": "words meaning group / order / arithmetic",
}
for part, (weight, full) in DIFFICULTY_WEIGHTS.items():
    print(f"{part:<13}{weight:>8.2f}{full:>9}   {source[part]}")

print()
print("the words that count as hard:")
print("  " + ", ".join(w.strip() for w in HARD_WORDS))
print()
print("A question needing four tables, three resolved values and three of those words")
print("scores 1.00. Under 0.35 starts at the small model, under 0.70 at the medium,")
print("above that at the large.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">4.</span> The ten questions, scored</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">One per difficulty band and each a different shape of query, so the comparison runs across a spread rather than ten repetitions of one question.</div></div>

In [ ]:
from nl2sql.nlp.understand import understand
from nl2sql.optimize.benchmark import load_questions
from nl2sql.optimize.variants import difficulty, difficulty_parts, starting_rung

# The ten are marked `demo: true` in data/questions.yaml.
graded = []
for item in (q for q in load_questions() if q.get("demo")):
    u = understand(item["question"])
    graded.append((difficulty(u), difficulty_parts(u), starting_rung(difficulty(u)), item))
graded.sort(key=lambda row: row[0])
questions = [item for _, _, _, item in graded]

print(f"{'#':<3}{'score':>7}{'tables':>8}{'values':>8}{'hard':>6}{'starts at':>11}   question")
print("-" * 112)
for position, (score, parts, rung, item) in enumerate(graded, 1):
    print(f"{position:<3}{score:>7.2f}{parts['tables']:>8}{parts['values']:>8}"
          f"{parts['hard words']:>6}{rung:>11}   {item['question'][:58]}")

print()
print(f"{len(questions)} questions, spanning {graded[0][0]:.2f} to {graded[-1][0]:.2f}.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">5.</span> How an answer is judged</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Not by comparing SQL text. Each query is run, and its rows are compared with the rows the reference query returns.</div></div>

In [ ]:
from nl2sql.optimize.benchmark import FLOAT_FIGURES, fingerprint_result

item = questions[2]
print("question :", item["question"])
print("reference:", " ".join(item["sql"].split())[:150])
print()
print("Correct means the same rows, in any order, whatever the wording. These two")
print("queries differ completely as text and count as the same answer:")
a = "SELECT COUNT(DISTINCT hospitalid) FROM hospital"
b = "SELECT COUNT(*) FROM (SELECT DISTINCT hospitalid FROM hospital)"
print(f"  {fingerprint_result(a)}   {a}")
print(f"  {fingerprint_result(b)}   {b}")
print()
print(f"Floats are compared to {FLOAT_FIGURES} significant figures, so an average reached by")
print("dividing a SUM by a COUNT matches one reached by AVG.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">6.</span> Running them</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Ten questions through five methods: fifty runs, each printed as it finishes.</div></div>

In [ ]:
from nl2sql.optimize.benchmark import compare

report = compare(questions=questions)

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">7.</span> Which method got which question right</div></div>

In [ ]:
variants = [row["variant"] for row in report["table"]]
answers = {}
for r in report["results"]:
    answers.setdefault(r["question"], {})[r["variant"]] = r


def mark(result):
    if result is None or result["correct"] is None:
        return "?"
    if result["correct"]:
        return "ok"
    return "refused" if result["refusal"] else "wrong"


print(f"{'score':>6}  {'question':<46}" + "".join(f"{v[:9]:>10}" for v in variants))
print("-" * (54 + 10 * len(variants)))
for score, _, _, item in graded:
    row = answers.get(item["question"], {})
    print(f"{score:>6.2f}  {item['question'][:45]:<46}"
          + "".join(f"{mark(row.get(v)):>10}" for v in variants))

print()
print(f"correct out of {len(questions)}:")
for v in variants:
    got = sum(1 for item in questions
              if (answers.get(item["question"], {}).get(v) or {}).get("correct"))
    print(f"  {v:<11}{got}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">8.</span> Where all five failed together</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">A question every method gets wrong is not evidence about the methods. They share stage 1, so it points at what happens before any of them is reached.</div></div>

In [ ]:
shared = [item for item in questions
          if all((answers.get(item["question"], {}).get(v) or {}).get("correct") is False
                 for v in variants)]

print(f"{len(shared)} of {len(questions)} questions were wrong for all five methods:")
for item in shared:
    print("   ", item["question"])

print()
print("Those cannot separate one method from another - every method inherits the same")
print("resolved values and the same schema from stage 1. What separates the methods is")
print("only the questions where they disagree:")
split = [item for item in questions
         if len({(answers.get(item["question"], {}).get(v) or {}).get("correct")
                 for v in variants}) > 1]
for item in split:
    row = answers[item["question"]]
    winners = [v for v in variants if (row.get(v) or {}).get("correct")]
    print(f"    {item['question'][:56]:<58} only {', '.join(winners) or 'none'}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">9.</span> The ranking on accuracy</div></div>

In [ ]:
basis = report["scoring"]
scored = report["with_reference_query"] + report["scored_by_consensus"]
print(f"ranked by {'agreement with the other methods' if basis == 'consensus' else basis}"
      f", over {scored} scored question(s)")
print()
for position, (name, score) in enumerate(report["ranking"], 1):
    print(f"  {position}. {name:<11} {score:.0%}")

spread = {score for _, score in report["ranking"]}
if len(spread) == 1:
    print()
    print("All five scored the same. Accuracy cannot pick a winner here, so the decision")
    print("falls to what each one spent getting there - the next two steps.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">10.</span> What each one spent</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Ranking on accuracy alone picks the most expensive answer. These are the columns that decide which method is worth running.</div></div>

In [ ]:
header = list(report["table"][0])
print(" | ".join(f"{h:>17}" for h in header))
print("-" * (20 * len(header)))
for row in report["table"]:
    print(" | ".join(f"{str(row[h]):>17}" for h in header))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">11.</span> Why each one landed where it did</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Read off the run: where it failed, how often it climbed, what it spent.</div></div>

In [ ]:
notes = {
    "baseline": "the whole schema, the large model, one call - the thing to beat",
    "lean": "only the columns stage 1 named; a wrong column list cannot be recovered from",
    "fewshot": "three solved questions prepended, paid for in prompt tokens on every call",
    "cascade": "starts at the rung the difficulty score picked, climbs on refusal or low confidence",
    "consensus": "three cheap samples, keeps the answer whose rows agree - three times the calls",
}
for row in report["table"]:
    v = row["variant"]
    score = row.get("accuracy", row.get("agreement", 0.0))
    failures = report["failures"].get(v) or {}
    print(f"{v:<11}{score:>6.0%}  {row['tokens/question']:>5} tok/q  "
          f"${row['dollars/100q']:<8} per 100q  {row['escalations']:>2} climbed")
    print(f"           {notes[v]}")
    if failures:
        print("           failed at: " + ", ".join(f"{k} x{n}" for k, n in failures.items()))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">12.</span> Which model actually answered</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">cascade is the only method that chooses. This is what it picked for each question, and whether the first choice held.</div></div>

In [ ]:
print(f"{'score':>6}  {'question':<44}{'model that answered':<34}{'climbed':>8}{'calls':>7}")
print("-" * 101)
for score, _, rung, item in graded:
    r = answers.get(item["question"], {}).get("cascade")
    if r is None:
        continue
    print(f"{score:>6.2f}  {item['question'][:43]:<44}{(r['model'] or '-'):<34}"
          f"{str(bool(r['escalated'])):>8}{r['calls']:>7}")

climbed = sum(1 for item in questions
              if (answers.get(item["question"], {}).get("cascade") or {}).get("escalated"))
print()
print(f"climbed on {climbed} of {len(questions)}. Where that is 0 the rung the difficulty score")
print("chose was already enough, and the ladder above it was never needed.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">13.</span> Where to put the escalation threshold</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Not a preference: the point that best separates the answers that turned out right from the ones that turned out wrong.</div></div>

In [ ]:
import json

from nl2sql.optimize.benchmark import calibrate

print(json.dumps(calibrate(), indent=1))
print()
print("`separates` is how cleanly perplexity divides right from wrong; 0.5 is a coin")
print("toss. Near 0.5 means confidence is not what carries the routing, and cascade's")
print("result comes from the difficulty score and the model ladder instead.")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">14.</span> The winner</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Most accurate first; where accuracy ties, the one that spent least getting there.</div></div>

In [ ]:
def score_of(row):
    return row.get("accuracy", row.get("agreement", 0.0))


order = sorted(report["table"],
               key=lambda r: (-score_of(r), r["dollars/100q"], r["tokens/question"]))
best, runner_up = order[0], order[1]

print(f"{'#':<3}{'method':<12}{'accuracy':>10}{'$/100 questions':>18}{'tokens/q':>10}{'median ms':>11}")
print("-" * 64)
for position, row in enumerate(order, 1):
    print(f"{position:<3}{row['variant']:<12}{score_of(row):>10.0%}"
          f"{row['dollars/100q']:>18}{row['tokens/question']:>10}{row['median ms']:>11}")

print()
print(f"Winner: {best['variant']}")
saved = 1 - (best["dollars/100q"] / runner_up["dollars/100q"]) if runner_up["dollars/100q"] else 0
if score_of(best) == score_of(runner_up):
    print(f"  Same accuracy as {runner_up['variant']} at {saved:.0%} less cost per hundred")
    print("  questions, so it is the one to run.")
else:
    print(f"  Ahead of {runner_up['variant']} on accuracy outright.")
print()
print("Read this against step 8: accuracy is capped by what stage 1 resolves, which")
print("every method shares. The cost column is where they genuinely differ today.")